# Subset allele metadata

In [1]:
import polars as pl
import pandas as pd

In [2]:
meta_inputs = "../1_inputs"
meta_outputs = "../3_outputs"

## VarChAMP Allele QC

Check the consistency between nucleotide change and its respective AA change

In [21]:
allele_df = pl.scan_csv(f"{meta_inputs}/VarChamp_Consolidated_allele_collection.tsv", separator="\t", infer_schema_length=10000)
allele_df = allele_df.collect()
allele_df

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""887G>A""","""Arg296Gln""","""ENSG00000177628""","""1""",155237453,"""NC_000001.11:155237453:C:T""","""ALE00000003""","""CCSBVarC000003""","""113""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""D12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
2,"""GBA""","""1448T>C""","""Leu483Pro""","""ENSG00000177628""","""1""",155235252,"""NC_000001.11:155235252:A:G""","""ALE00000004""","""CCSBVarC000004""","""231""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""E12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…"
25,"""SLC9A3R1""","""328C>G""","""Leu110Val""","""ENSG00000109062""","""17""",74749174,"""NC_000017.11:74749174:C:G""","""ALE00000005""","""CCSBVarC000005""","""252""","""200008""","""NULL""","""GDEhDisVCh_40045""","""E04""","""RC4""","""ATGAGCGCGGACGCAGCGGCCGGGGCGCCC…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
8999,"""TTR""","""355G>A""","""Asp119Asn""","""ENSG00000118271""","""18""",31598586,"""NC_000018.10:31598586:G:A""","""ALE00008468""","""CCSBVarC008481""","""NULL""","""NULL""","""390""","""NULL""","""NULL""","""Edgotyping3""","""ATGGCTTCTCATCGTCTGCTCCTCCTCTGC…"
7323,"""PMP22""","""152A>G""","""His51Arg""","""ENSG00000109099""","""17""",15259120,"""NC_000017.11:15259120:T:C""","""ALE00008469""","""CCSBVarC008466""","""NULL""","""NULL""","""394""","""NULL""","""NULL""","""Edgotyping3""","""ATGCTCCTCCTGTTGCTGAGTATCATCGTC…"
2651,"""CLN6""","""89G>C""","""Gly30Ala""","""ENSG00000128973""","""15""",68218645,"""NC_000015.10:68218645:C:G""","""ALE00008470""","""CCSBVarC008436""","""NULL""","""NULL""","""436""","""NULL""","""NULL""","""Edgotyping3""","""ATGGAGGCGACGCGGAGGCGGCAGCACCTG…"


In [29]:
## QC
from Bio.Seq import Seq
aa_single_to_three = {
    'A': 'Ala',  # Alanine
    'R': 'Arg',  # Arginine
    'N': 'Asn',  # Asparagine
    'D': 'Asp',  # Aspartic acid
    'C': 'Cys',  # Cysteine
    'Q': 'Gln',  # Glutamine
    'E': 'Glu',  # Glutamic acid
    'G': 'Gly',  # Glycine
    'H': 'His',  # Histidine
    'I': 'Ile',  # Isoleucine
    'L': 'Leu',  # Leucine
    'K': 'Lys',  # Lysine
    'M': 'Met',  # Methionine
    'F': 'Phe',  # Phenylalanine
    'P': 'Pro',  # Proline
    'S': 'Ser',  # Serine
    'T': 'Thr',  # Threonine
    'W': 'Trp',  # Tryptophan
    'Y': 'Tyr',  # Tyrosine
    'V': 'Val'   # Valine
}

for row in allele_df.iter_rows(named=True): #.filter(pl.col("hgvs_nt").is_duplicated()).sort(by="hgvs_nt")
    # hgvs_df = dat_info.filter(pl.col("hgvs_nt")==hgvs_nt)
    nuc_pos = int(row["nt_change"][:-3])
    nuc_r, nuc_v = row["nt_change"].split(">")[0][-1], row["nt_change"].split(">")[-1][-1]

    nuc_var = row["sequence"][nuc_pos-1] ## row["ref_cds"][nuc_pos-1], 
    nuc_seq = row["sequence"]
    # print(nuc_r, nuc_v, nuc_ref, nuc_var)
    prot_pos = int(row["aa_change"][3:-3])
    prot_r, prot_v = row["aa_change"][:3], row["aa_change"][-3:]
    prot_ref_seq = Seq(nuc_seq[:nuc_pos-1]+nuc_r+nuc_seq[nuc_pos:]).translate()
    prot_var_seq = Seq(nuc_seq).translate()
    try:
        prot_ref, prot_var = aa_single_to_three[prot_ref_seq[prot_pos-1]], aa_single_to_three[prot_var_seq[prot_pos-1]]
        # print(prot_r, prot_v, prot_ref, prot_var)
        if nuc_v != nuc_var or prot_r != prot_ref or prot_v != prot_var:
            print("Original annotations:", row["aa_change"], row["nt_change"])
            print(f"Actual Nuc Var:{nuc_var} | " + \
                f"Actual Prot Ref:{prot_ref}, Var:{prot_var}")
            print("========================================================")
    except:
        print(row["aa_change"], row["nt_change"], "| Protein seq len ONLY:", len(prot_ref_seq))
        print("========================================================")
    # break


## Create gene-level annotation file

In [5]:
# get unique info for genes
gene_cols = ["orf_id", "symbol", "ensembl_gene_id", "chr"]
gene_info = pd.read_table(f"{meta_inputs}/VarChamp_Consolidated_allele_collection.tsv")[gene_cols]
gene_info = gene_info.drop_duplicates()
duplicate_orfs = gene_info[gene_info.duplicated(subset=['symbol'], keep=False)]

In [ ]:
# Need to figure out why there are duplicated ORF IDs
duplicate_orfs = gene_info[gene_info.duplicated(subset=['symbol'], keep=False)]

In [ ]:
# write out file
gene_info.to_csv(f"{meta_outputs}/orf_info.csv", index=False)

## Pivot metadata with respect to mut_ids

In [5]:
# process metadata
metadata = pl.DataFrame(pd.read_csv(f"{meta_outputs}/varchamp_consolidated_alleles_clinvar_gnomad.tsv", sep="\t"))
mut_cols = ["mutation_id_rc4", "mutation_id_cegs2", "mutation_id_edgotyping3"]
id_vars = [col for col in metadata.columns if col not in mut_cols]
metadata = metadata.unpivot(index=id_vars, 
                            on=mut_cols,
                            variable_name="mutation_collection",
                            value_name="mut_id")

priority_col = ["orf_id", "mut_id", "symbol", "aa_change", "nt_change", "ensembl_gene_id", "collection", "clinvar_clnsig_clean", "gnomad_af", "StarStatus"]
metadata = metadata.select(priority_col).filter(~pl.col("mut_id").is_null()).with_columns(
    pl.col("mut_id").cast(pl.Int64).alias("mut_id"),
    pl.col("orf_id").cast(pl.Int64).alias("orf_id"),
    pl.concat_str([pl.col("symbol"), pl.col("aa_change")], separator="_").alias("allele_0")
).unique()

metadata.write_csv(f"{meta_outputs}/slim_metadata.csv")